In [ ]:
import Pkg; Pkg.add("ITensors")

In [ ]:
using DelimitedFiles
using Pkg
Pkg.activate("../")  # Activate the main project
using Dleto
using Random
using ITensors
using Statistics
using Plots
using Combinatorics

In [ ]:

function process_high_school_hypergraph(k::Int; seed::Union{Int,Nothing}=nothing, number_label::Union{Int,String}=1)
  
    # Load the contact-high-school temporal hypergraph dataset


    # Load the three data files
    nverts_file = "contact-high-school/contact-high-school-nverts.txt"
    simplices_file = "contact-high-school/contact-high-school-simplices.txt"
    times_file = "contact-high-school/contact-high-school-times.txt"

    # Read number of vertices per simplex
    nverts_full = vec(readdlm(nverts_file, Int))
    println("Total number of simplices: ", length(nverts_full))

    # Read all simplex node IDs (contiguous list)
    all_nodes_full = vec(readdlm(simplices_file, Int))
    println("Total node entries: ", length(all_nodes_full))

    # Read timestamps
    timestamps_full = vec(readdlm(times_file, Int))
    println("Total timestamps: ", length(timestamps_full))

    # Filter to random k nodes only
    all_unique_nodes = sort(unique(all_nodes_full))
    if seed !== nothing
        Random.seed!(seed)
        println("Random seed set to $seed")
    end
    random_k_nodes = Set(shuffle(all_unique_nodes)[1:k])
    println("Filtering to random $k nodes: $(sort(collect(random_k_nodes)))")

    # Filter simplices to only include those with all nodes in random_k_nodes
    filtered_indices = Int[]
    node_idx = 1
    for (simplex_idx, nvert) in enumerate(nverts_full)
        simplex_nodes = all_nodes_full[node_idx:(node_idx + nvert - 1)]
        if all(node in random_k_nodes for node in simplex_nodes)
            push!(filtered_indices, simplex_idx)
        end
        node_idx += nvert
    end

    # Create filtered datasets
    nverts = nverts_full[filtered_indices]
    timestamps = timestamps_full[filtered_indices]

    # Rebuild all_nodes for filtered simplices
    all_nodes = Int[]
    for simplex_idx in filtered_indices
        nvert = nverts_full[simplex_idx]
        orig_pos = sum(nverts_full[1:simplex_idx-1]) + 1
        simplex_nodes = all_nodes_full[orig_pos:(orig_pos + nvert - 1)]
        append!(all_nodes, simplex_nodes)
    end
    max_view = length(unique(all_nodes))
    println("\nFiltered dataset statistics:")
    println("Number of timestamped simplices: ", length(nverts))
    println("Number of unique node IDs: ", length(unique(all_nodes)))
    println("Simplex size distribution:")
    size_counts = Dict{Int, Int}()
    for size in nverts
        size_counts[size] = get(size_counts, size, 0) + 1
    end
    for size in sort(collect(keys(size_counts)))
        count = size_counts[size]
        println("  Size $size: $count simplices")
    end

    # Parse simplices from the contiguous node list
    simplices = Vector{Vector{Int}}()
    node_idx = 1
    for nvert in nverts
        simplex_nodes = all_nodes[node_idx:(node_idx + nvert - 1)]
        push!(simplices, simplex_nodes)
        node_idx += nvert
    end
    println("Parsed ", length(simplices), " simplices")
    println("First few simplices:")
    for i in 1:min(10, length(simplices))
        println("  Simplex $i (time $(timestamps[i])): $(simplices[i])")
    end

    # Create simple temporal simplex structure for processing
    temporal_simplices = [(simplices[i], timestamps[i], nverts[i]) for i in 1:length(simplices)]
    println("Created temporal hypergraph with ", length(temporal_simplices), " temporal simplices")

    # Basic statistics
    all_unique_nodes = sort(unique(all_nodes))
    println("Number of nodes: ", length(all_unique_nodes))
    println("Time range: $(minimum(timestamps)) to $(maximum(timestamps))")
    println("\nReady for tensor creation!")

    # Create a 3-mode tensor by decomposing hyperedges into 3-node subsets
    println("Creating 3-mode tensor from hyperedge decomposition ($k nodes only)...")
    unique_nodes = all_unique_nodes[1:max_view]
    node_to_idx = Dict(node => i for (i, node) in enumerate(unique_nodes))
    n_nodes = max_view
    println("Tensor dimensions:")
    println("  Node 1: $n_nodes")
    println("  Node 2: $n_nodes")
    println("  Node 3: $n_nodes")
    println("  Total entries: $(n_nodes^3)")
    println("  Using nodes: $(minimum(unique_nodes)) to $(maximum(unique_nodes))")

    function get_3node_combinations(nodes)
        if length(nodes) < 3
            return []
        elseif length(nodes) == 3
            return [nodes]
        else
            combinations = []
            for i in 1:length(nodes)-2
                for j in i+1:length(nodes)-1
                    for k in j+1:length(nodes)
                        push!(combinations, [nodes[i], nodes[j], nodes[k]])
                    end
                end
            end
            return combinations
        end
    end

    # Create ITensor indices for the 3-mode tensor
    i = Index(n_nodes, "node1")
    j = Index(n_nodes, "node2")
    k = Index(n_nodes, "node3")
    Triangles = ITensor(Float32, i, j, k)

    # Populate the tensor by counting 3-node combinations
    processed_simplices = 0
    total_triangles = 0
    skipped_small = 0
    for (idx, ts) in enumerate(temporal_simplices)
        triangle_combinations = get_3node_combinations(ts[1]) # ts[1] is the nodes vector
        if isempty(triangle_combinations)
            skipped_small += 1
            continue
        end
        for triangle in triangle_combinations
            node_indices = []
            for node in triangle
                if node in keys(node_to_idx) # Double check that all nodes are within our random sample
                    push!(node_indices, node_to_idx[node])
                end

            end
            if length(node_indices) == 3
                a, b, c = node_indices[1], node_indices[2], node_indices[3]
                Triangles[i=>a, j=>b, k=>c] += 1.0
                Triangles[i=>a, j=>c, k=>b] += 1.0
                Triangles[i=>b, j=>a, k=>c] += 1.0
                Triangles[i=>b, j=>c, k=>a] += 1.0
                Triangles[i=>c, j=>a, k=>b] += 1.0
                Triangles[i=>c, j=>b, k=>a] += 1.0
                total_triangles += 1
            end
        end
        processed_simplices += 1
    end

    
    # After populating Triangles and before tensor analysis
    # Output filtered hyperedges (unique, no permutations)
    hyperedges_filename = "filtered_hyperedges_$(string(number_label)).txt" 
    written = Set{Tuple{Int,Int,Int}}()   
    open(hyperedges_filename, "w") do io
        for a in 1:n_nodes, b in 1:n_nodes, c in 1:n_nodes
            if Triangles[i=>a, j=>b, k=>c] == 1.0
                edge = sort((a, b, c)) # sort to ignore permutations
                if !(edge in written)
                    print(io, "[", join(edge, ","), "]; ")
                    push!(written, edge)
                end
            end
        end
    end
    println("Filtered hyperedges saved to $hyperedges_filename")

    # Output all permutations of each unique hyperedge
    permuted_filename = "filtered_hyperedges_permuted_$(string(number_label)).txt"
    open(permuted_filename, "w") do io
        for edge in written
            perms = collect(permutations(edge))
            for p in perms
                print(io, "(", join(p, ","), "); ")
            end
        end
    end
    println("Permuted hyperedges saved to $permuted_filename")




    # Analyze the tensor
    T_array = Array(Triangles, i, j, k)
    non_zero_entries = count(x -> x != 0, T_array)
    total_entries = length(T_array)
    println("✅ 3-mode triangle tensor created!")
    println("Processed simplices: $processed_simplices")
    println("Skipped simplices (< 3 nodes): $skipped_small")
    println("Total triangles extracted: $total_triangles")
    println("Non-zero tensor entries: $non_zero_entries / $total_entries")
    println("Sparsity: $(round((total_entries - non_zero_entries) / total_entries * 100, digits=1))%")
    println("Max triangle count: $(Int(maximum(T_array)))")
    println("Mean triangle count: $(round(mean(T_array[T_array .> 0]), digits=2))")

    # Plot the tensor
    # plot_tensor(Triangles, title="High School Contact Triangle Tensor")

    # Create a k×k×k subsample tensor for faster processing
    println("Creating $k×$k×$k subsample tensor...")
    max = max_view
    i_small = Index(max, "node1_small")
    j_small = Index(max, "node2_small")
    k_small = Index(max, "node3_small")
    SmallHigh = ITensor(i_small, j_small, k_small)
    T_small_array = T_array[1:max, 1:max, 1:max]
    for a in 1:max, b in 1:max, c in 1:max
        if T_small_array[a, b, c] != 0
            SmallHigh[i_small=>a, j_small=>b, k_small=>c] = T_small_array[a, b, c]
        end
    end
    small_non_zero = count(x -> x != 0, T_small_array)
    small_total = length(T_small_array)
    println("✅ Subsample tensor created!")

    # Perform stratified decomposition on the subsample
    println("\n🔬 Performing stratified decomposition on SmallHigh tensor...")
    @time SmallHigh_strat, Xs_strat_small = stratify(SmallHigh)
    println("✅ SmallHigh stratification complete!")

    # Plot SmallHigh and SmallHigh_strat side by side
    p1 = plot_tensor(SmallHigh; title="Original SmallHigh", color=:blue)
    p2 = plot_tensor(SmallHigh_strat; title="Stratified SmallHigh", color=:red)
    Plots.plot(p1, p2; layout=(1,2), size=(800,400))

end

In [ ]:
process_high_school_hypergraph(80; number_label=1)

In [ ]:
process_high_school_hypergraph(80; number_label=2)

In [ ]:
process_high_school_hypergraph(80; number_label=3)

In [ ]:
process_high_school_hypergraph(80; number_label=4)

In [ ]:
process_high_school_hypergraph(80; number_label=5)

In [ ]:
# process_high_school_hypergraph(60)

In [ ]:
# process_high_school_hypergraph(90)

In [ ]:
# process_high_school_hypergraph(90)

In [ ]:
# process_high_school_hypergraph(90)

In [ ]:
# process_high_school_hypergraph(150)

In [ ]:
# process_high_school_hypergraph(150)

In [ ]:
# process_high_school_hypergraph(150)

In [ ]:
# process_high_school_hypergraph(200)

In [ ]:
# process_high_school_hypergraph(200)

In [ ]:
# process_high_school_hypergraph(200)